# 🌱 FarmerBud · 2. Quali colture seminare nel mio campo?

*Un aiuto per i piccoli agricoltori della Pianura Padana (Piemonte, Lombardia, Veneto, Emilia-Romagna).*

**Come si usa**
1. Esegui tutte le celle in ordine (su Colab: *Runtime → Esegui tutte*).
2. Nella mappa in fondo **clicca sul tuo campo**: le coordinate vengono prese dal clic.
3. Se vuoi, indica la coltura dell'anno scorso, se puoi irrigare e i dati della tua analisi del terreno.
4. Premi **Calcola**: dopo circa 30 secondi compaiono le 5 colture più adatte e i consigli.

**Cosa fa dietro le quinte**
- Scarica 20 anni di clima giornaliero del punto (NASA POWER, 2006-2025) e il suolo (SoilGrids).
- Per ogni coltura e per ognuno dei 20 anni calcola le stesse variabili usate per addestrare le reti.
- Usa le **4 reti neurali** (`modelli.joblib`, creato da `1_addestramento_reti_IT.ipynb`): resa, acqua, azoto, stabilità.
- La resa attesa è la mediana dei 20 anni; l'acqua "anno secco" è quella che serve almeno 1 anno su 5.
- Ordina le colture e scrive i consigli con regole semplici (rotazioni, pH, acqua).

**File necessario:** `modelli.joblib`. Se non è nella cartella del notebook (per esempio su Colab), viene scaricato da solo dal repository GitHub.

**Su GitHub la mappa interattiva non si vede:** in fondo (sezione 7) c'è un esempio già calcolato su un campo vero.

> ⚠️ Prototipo per un hackathon: **non è un consiglio agronomico**. Prima di seminare, parlane con un tecnico.

## 0. Librerie

In [1]:
# subprocess e sys servono per installare una libreria se manca.
import subprocess
import sys

# Prova a importare ipyleaflet (la mappa); se manca, la installa con pip.
try:
    import ipyleaflet
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ipyleaflet"])

# Su Google Colab le mappe interattive vanno attivate; fuori da Colab questo passo viene saltato.
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

# io legge un testo come se fosse un file; os controlla se un file esiste; re cerca pezzi di testo; time fa delle pause.
import io
import os
import re
import time

# urllib.request scarica un file da internet (le reti, se mancano).
import urllib.request

# ipywidgets crea pulsanti, menu e caselle; lo chiamiamo "widgets".
import ipywidgets as widgets

# joblib legge il file con le 4 reti.
import joblib

# numpy fa calcoli su liste di numeri; pandas maneggia tabelle; requests scarica dati da internet.
import numpy as np
import pandas as pd
import requests

# sklearn (scikit-learn) è la libreria delle reti: ci serve il suo numero di versione.
import sklearn

# Dalla libreria della mappa: la mappa, il segnaposto, i cerchietti e il rettangolo.
from ipyleaflet import CircleMarker, Map, Marker, Rectangle

# display mostra tabelle e testo formattato nel notebook; HTML scrive testo con grassetti ed elenchi.
from IPython.display import HTML, display

## 1. Le 4 reti e le tabelle delle colture
`modelli.joblib` contiene le 4 reti, la resa mediana di ogni coltura e i 33 punti di pianura usati per addestrare.
Le tabelle sotto sono scritte a mano (fonti in `01_Datasets/tabelle/FONTI.md`).

In [2]:
# Il repository GitHub del progetto, da cui scaricare le reti se mancano.
REPO = "byterunnersAI/FarmerBud"

# Se modelli.joblib non è nella cartella (per esempio su Colab), lo scarica da GitHub.
if not os.path.exists("modelli.joblib"):
    urllib.request.urlretrieve(f"https://raw.githubusercontent.com/{REPO}/main/02_AI_Model/modelli.joblib", "modelli.joblib")

# Legge il file con le 4 reti.
models = joblib.load("modelli.joblib")

# Reti salvate con un'altra versione di scikit-learn di solito funzionano, ma è meglio saperlo.
if models["sklearn_version"] != sklearn.__version__:
    print(f"⚠️ Reti create con scikit-learn {models['sklearn_version']}, qui c'è la {sklearn.__version__}. "
          "Se qualcosa non va, esegui prima il notebook 1 per ricrearle.")

# Tabella dei 33 punti di pianura (nome, coordinate, suolo).
provinces = models["provinces"]

# Le colture. Per ognuna: nome; stagione per il clima (mese di inizio e fine, se inizio > fine parte l'anno prima);
# base dei gradi-giorno (°C); data di semina e durata del ciclo (per l'acqua); pH ottimale e limite; suoli adatti.
CROPS = {
    "COMMEAT": dict(name="frumento tenero", season=(10, 6), gdd_base=0, sowing="10-25", days=240, ph=(6.0, 7.0, 5.5, 8.5), soils="media"),
    "WHEATD": dict(name="frumento duro", season=(10, 6), gdd_base=0, sowing="10-25", days=240, ph=(6.0, 7.0, 5.5, 8.5), soils="pesante, media"),
    "BARLEY": dict(name="orzo", season=(10, 6), gdd_base=0, sowing="10-15", days=235, ph=(6.5, 7.5, 6.0, 8.0), soils="media"),
    "RAPE": dict(name="colza", season=(9, 6), gdd_base=0, sowing="09-15", days=270, ph=(6.5, 7.6, 5.5, 8.0), soils="media, leggera"),
    "MAIZE": dict(name="mais", season=(4, 9), gdd_base=10, sowing="04-10", days=160, ph=(5.0, 7.0, 4.5, 8.5), soils="media"),
    "SOYA": dict(name="soia", season=(5, 9), gdd_base=10, sowing="05-10", days=140, ph=(5.5, 6.5, 4.5, 8.4), soils="media"),
    "SUNFLO": dict(name="girasole", season=(4, 8), gdd_base=10, sowing="04-15", days=130, ph=(6.0, 7.5, 5.5, 8.0), soils="media, leggera"),
    "SORGH": dict(name="sorgo", season=(5, 9), gdd_base=10, sowing="05-15", days=130, ph=(5.5, 7.5, 5.0, 8.0), soils="pesante, media"),
    "RICE": dict(name="riso", season=(4, 9), gdd_base=10, sowing="04-20", days=150, ph=(5.5, 7.0, 4.5, 9.0), soils="pesante, media, leggera"),
    "BEETS": dict(name="barbabietola da zucchero", season=(3, 9), gdd_base=5, sowing="03-15", days=170, ph=(6.0, 6.8, 5.5, 8.5), soils="pesante, media"),
}

# Coltura dell'anno scorso: le 10 colture più l'erba medica (che non prevediamo, ma conta per la rotazione).
PREVIOUS_OPTIONS = [("non so / altro", None)] + [(c["name"], code) for code, c in CROPS.items()] + [("erba medica", "LUCERNE")]

# Coppie (coltura, coltura dell'anno prima) VIETATE dai disciplinari, con il motivo.
FORBIDDEN = {
    ("BEETS", "RAPE"): "nematode della barbabietola (Heterodera schachtii), comune alle crucifere",
    ("RAPE", "BEETS"): "nematode della barbabietola (Heterodera schachtii)",
    ("RAPE", "SOYA"): "Sclerotinia, malattia comune a colza, soia e girasole",
    ("RAPE", "SUNFLO"): "Sclerotinia, malattia comune a colza, soia e girasole",
    ("SOYA", "RAPE"): "Sclerotinia, malattia comune a colza, soia e girasole",
    ("SOYA", "SUNFLO"): "Sclerotinia, malattia comune a colza, soia e girasole",
    ("SUNFLO", "RAPE"): "Sclerotinia, malattia comune a colza, soia e girasole",
    ("SUNFLO", "SOYA"): "Sclerotinia, malattia comune a colza, soia e girasole",
    ("WHEATD", "WHEATD"): "il frumento duro non può seguire un cereale autunnale (fusariosi)",
    ("WHEATD", "COMMEAT"): "il frumento duro non può seguire un cereale autunnale (fusariosi)",
    ("WHEATD", "BARLEY"): "il frumento duro non può seguire un cereale autunnale (fusariosi)",
    ("RAPE", "RAPE"): "ristoppio non ammesso",
    ("SOYA", "SOYA"): "ristoppio non ammesso",
    ("SUNFLO", "SUNFLO"): "ristoppio non ammesso",
    ("BEETS", "BEETS"): "nematode: la barbabietola torna sullo stesso campo solo dopo 3 anni",
}

# Coppie consentite ma con ATTENZIONE, con il motivo.
WARNINGS = {
    ("MAIZE", "MAIZE"): "mais dopo mais: ammesso una sola volta in 5 anni; la rotazione basta a contenere la diabrotica",
    ("COMMEAT", "COMMEAT"): "ristoppio ammesso una sola volta in 5 anni (fusariosi e mal del piede)",
    ("COMMEAT", "BARLEY"): "dopo l'orzo conta come ristoppio (fusariosi)",
    ("COMMEAT", "WHEATD"): "dopo il frumento duro conta come ristoppio (fusariosi)",
    ("BARLEY", "BARLEY"): "ristoppio ammesso una sola volta in 5 anni",
    ("BARLEY", "COMMEAT"): "dopo il frumento conta come ristoppio",
    ("BARLEY", "WHEATD"): "dopo il frumento conta come ristoppio",
    ("SORGH", "SORGH"): "ristoppio ammesso una sola volta in 5 anni",
    ("BEETS", "LUCERNE"): "elateridi: meglio aspettare 2 anni dopo un medicaio",
    ("MAIZE", "LUCERNE"): "elateridi: rompi il medicaio l'estate prima o subito prima della semina",
    ("WHEATD", "MAIZE"): "fusariosi: interra i residui del mais con un'aratura",
    ("WHEATD", "SORGH"): "fusariosi: interra i residui del sorgo con un'aratura",
}

# Quanto "costa" l'acqua nella classifica: punti di resa relativa tolti ogni 100 mm di irrigazione (scelta del team).
WATER_WEIGHT_IRRIGATED = 0.05
WATER_WEIGHT_RAINFED = 0.15

## 2. Scaricare clima, quota e suolo del punto
Le correzioni sono le stesse usate per creare i dati di addestramento (`01_Datasets/s3_clima.py`): se fossero
diverse, le reti riceverebbero numeri a cui non sono abituate.
- **Quota** dal servizio Open-Meteo: serve per correggere la temperatura e per l'ET0.
- **Clima** da NASA POWER: celle di circa 50 km, temperatura corretta per la differenza di quota (calibrata su Open-Meteo).
- **ET0** (quanta acqua "chiede" l'aria) con la formula FAO-56 Penman-Monteith.
- **Suolo** da SoilGrids (media 0-30 cm). I centri abitati non hanno dati: proviamo qualche punto vicino e,
  se non risponde, usiamo il suolo del punto di pianura più vicino.

In [3]:
# Restituisce la quota (m) di un punto, dal servizio gratuito di Open-Meteo.
def get_elevation(lat, lon):
    answer = requests.get("https://api.open-meteo.com/v1/elevation", params={"latitude": lat, "longitude": lon}, timeout=30)
    return answer.json()["elevation"][0]


# ET0 giornaliera (mm) con FAO-56, equazione 6. Ogni riga è un pezzo della formula del manuale FAO.
def et0_penman_monteith(tmean, tmax, tmin, rh, u2, rs, lat, elevation, doy):
    # Pressione di vapore a saturazione a una temperatura t (kPa).
    e0 = lambda t: 0.6108 * np.exp(17.27 * t / (t + 237.3))
    # Pressione di vapore a saturazione media ed effettiva (dall'umidità relativa).
    es = (e0(tmax) + e0(tmin)) / 2
    ea = rh / 100 * es
    # Pendenza della curva di vapore e costante psicrometrica (dipende dalla pressione, quindi dalla quota).
    delta = 4098 * e0(tmean) / (tmean + 237.3) ** 2
    gamma = 0.000665 * 101.3 * ((293 - 0.0065 * elevation) / 293) ** 5.26
    # Radiazione che arriverebbe in cima all'atmosfera, da latitudine e giorno dell'anno.
    phi = np.radians(lat)
    dr = 1 + 0.033 * np.cos(2 * np.pi * doy / 365)
    decl = 0.409 * np.sin(2 * np.pi * doy / 365 - 1.39)
    ws = np.arccos(np.clip(-np.tan(phi) * np.tan(decl), -1, 1))
    ra = 24 * 60 / np.pi * 0.0820 * dr * (ws * np.sin(phi) * np.sin(decl) + np.cos(phi) * np.cos(decl) * np.sin(ws))
    # Radiazione netta: quella che entra (onde corte) meno quella che esce (onde lunghe).
    rso = (0.75 + 2e-5 * elevation) * ra
    rnl = 4.903e-9 * ((tmax + 273.16) ** 4 + (tmin + 273.16) ** 4) / 2 * (0.34 - 0.14 * np.sqrt(ea)) * (1.35 * np.clip(rs / rso, 0, 1) - 0.35)
    rn = 0.77 * rs - rnl
    # La formula finale di Penman-Monteith; l'ET0 non può essere negativa.
    et0 = (0.408 * delta * rn + gamma * 900 / (tmean + 273) * u2 * (es - ea)) / (delta + gamma * (1 + 0.34 * u2))
    return np.clip(et0, 0, None)


# Scarica il clima giornaliero dal 1 settembre 2005 al 31 dicembre 2025 da NASA POWER e lo corregge.
def download_climate(lat, lon, elevation):
    answer = requests.get("https://power.larc.nasa.gov/api/temporal/daily/point", timeout=180, params={
        "parameters": "T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,ALLSKY_SFC_SW_DWN,RH2M,WS2M",
        "community": "AG", "latitude": lat, "longitude": lon,
        "start": "20050901", "end": "20251231", "format": "CSV"})
    text = answer.text
    # Nell'intestazione del file c'è la quota media della cella di NASA POWER.
    cell_elevation = float(re.search(r"=\s*([\d.]+)\s*meters", text).group(1))
    # I dati veri iniziano dopo la riga "-END HEADER-"; -999 vuol dire dato mancante.
    daily = pd.read_csv(io.StringIO(text.split("-END HEADER-")[1].strip())).replace(-999, np.nan)
    # Costruisce la data da anno e giorno dell'anno (DOY).
    daily["date"] = pd.to_datetime(daily["YEAR"].astype(str) + daily["DOY"].astype(str).str.zfill(3), format="%Y%j")
    # Correzione della temperatura: -0,56 °C + 4,2 °C ogni 1000 m in cui la cella è più alta del campo.
    correction = -0.56 + (cell_elevation - elevation) * 0.64 * 6.5 / 1000
    for column in ["T2M", "T2M_MAX", "T2M_MIN"]:
        daily[column] = daily[column] + correction
    # Calcola l'ET0 di ogni giorno.
    daily["ET0"] = et0_penman_monteith(daily["T2M"], daily["T2M_MAX"], daily["T2M_MIN"], daily["RH2M"], daily["WS2M"],
                                       daily["ALLSKY_SFC_SW_DWN"], lat, elevation, daily["DOY"])
    return daily


# Interroga SoilGrids in un punto e restituisce pH, tessitura, carbonio e azoto (media 0-30 cm), o None.
def query_soilgrids(lat, lon):
    names = {"phh2o": "ph", "clay": "argilla_pct", "sand": "sabbia_pct", "silt": "limo_pct",
             "soc": "carbonio_organico_g_kg", "nitrogen": "azoto_g_kg"}
    # Spessore di ogni strato in cm: è il peso nella media.
    depths = {"0-5cm": 5, "5-15cm": 10, "15-30cm": 15}
    try:
        answer = requests.get("https://rest.isric.org/soilgrids/v2.0/properties/query", timeout=60,
                              params=[("lon", round(lon, 4)), ("lat", round(lat, 4)), ("value", "mean")]
                              + [("property", p) for p in names] + [("depth", d) for d in depths])
        layers = answer.json()["properties"]["layers"]
    except Exception:
        return None
    soil = {}
    for layer in layers:
        # Coppie (spessore, valore) degli strati che hanno un dato.
        values = [(depths[d["label"]], d["values"]["mean"]) for d in layer["depths"] if d["values"]["mean"] is not None]
        # Nessun dato (centro abitato o acqua): questo punto non va bene.
        if not values:
            return None
        # Media pesata, divisa per il "d_factor" di SoilGrids per avere le unità normali.
        soil[names[layer["name"]]] = sum(w * v for w, v in values) / sum(w for w, _ in values) / layer["unit_measure"]["d_factor"]
    return soil


# Cerca il suolo nel punto e, se manca, in 4 punti a circa 2 km; se non trova niente usa il punto di pianura più vicino.
def get_soil(lat, lon, nearest):
    for dlat, dlon in [(0, 0), (0.02, 0), (-0.02, 0), (0, 0.02), (0, -0.02)]:
        soil = query_soilgrids(lat + dlat, lon + dlon)
        if soil:
            return soil, "SoilGrids"
        # Breve pausa per non sovraccaricare il servizio.
        time.sleep(1)
    columns = ["ph", "argilla_pct", "sabbia_pct", "limo_pct", "carbonio_organico_g_kg", "azoto_g_kg"]
    return nearest[columns].to_dict(), f"punto di pianura più vicino ({nearest['comune_punto']})"

## 3. Dal clima giornaliero agli input delle reti
Ogni rete vuole gli input **con gli stessi nomi e lo stesso significato** che aveva in addestramento.
- **Rete della resa:** 13 variabili sul clima della stagione di quella coltura in quell'anno.
- **Rete dell'acqua:** ET0 e pioggia dalla semina al raccolto (pioggia divisa per 0,80, come nei dati di addestramento).
- **Rete dell'azoto:** la pioggia da ottobre a febbraio (sopra 300 mm l'acqua porta via azoto).
- **Rete della stabilità:** media e variabilità del clima della stagione nei 20 anni.

La funzione `predict` evita di ripetere codice: trasforma la coltura in colonne 0/1, mette le colonne
nell'ordine giusto (`reindex`) e chiede la previsione alla rete.

In [4]:
# La pioggia di NASA POWER è circa l'80% di quella di Open-Meteo: per l'acqua la riportiamo su (come in addestramento).
RAIN_RATIO = 0.80

# Anni del clima usati per simulare il campo.
YEARS = range(2006, 2026)


# Il periodo secco più lungo: quanti giorni di fila con meno di 1 mm di pioggia.
def longest_dry_spell(rain):
    best = current = 0
    for dry in (rain < 1.0):
        current = current + 1 if dry else 0
        best = max(best, current)
    return best


# Le 13 variabili del clima per una coltura e un anno di raccolto (uguali a 01_Datasets/s3_clima.py).
def season_climate(daily, code, year):
    start_month, end_month = CROPS[code]["season"]
    # Se il mese di inizio è dopo quello di fine, la stagione parte l'anno prima (colture autunnali).
    start = pd.Timestamp(year - 1 if start_month > end_month else year, start_month, 1)
    end = pd.Timestamp(year, end_month, 1) + pd.offsets.MonthEnd(0)
    s = daily[(daily["date"] >= start) & (daily["date"] <= end)]
    return dict(
        temp_media=s["T2M"].mean(), temp_max_media=s["T2M_MAX"].mean(), temp_min_media=s["T2M_MIN"].mean(),
        pioggia_mm=s["PRECTOTCORR"].sum(), et0_mm=s["ET0"].sum(),
        bilancio_idrico_mm=s["PRECTOTCORR"].sum() - s["ET0"].sum(),
        radiazione_MJ_m2=s["ALLSKY_SFC_SW_DWN"].sum(), umidita_media=s["RH2M"].mean(), vento_medio_m_s=s["WS2M"].mean(),
        gradi_giorno=np.clip(s["T2M"] - CROPS[code]["gdd_base"], 0, None).sum(),
        giorni_sopra_35=int((s["T2M_MAX"] > 35).sum()), giorni_gelo=int((s["T2M_MIN"] < 0).sum()),
        periodo_secco_max_giorni=longest_dry_spell(s["PRECTOTCORR"].values),
    )


# ET0 e pioggia dalla semina al raccolto, per la rete dell'acqua.
def cycle_water(daily, code, year):
    month, day = map(int, CROPS[code]["sowing"].split("-"))
    # Semine da agosto in poi sono autunnali: avvengono l'anno prima del raccolto.
    sowing = pd.Timestamp(year - 1 if month >= 8 else year, month, day)
    s = daily[(daily["date"] >= sowing) & (daily["date"] < sowing + pd.Timedelta(days=CROPS[code]["days"]))]
    return dict(et0_ciclo_mm=s["ET0"].sum(), pioggia_ciclo_mm=s["PRECTOTCORR"].sum() / RAIN_RATIO)


# Pioggia dal 1 ottobre al 28 febbraio prima del raccolto, per la rete dell'azoto.
def winter_rain(daily, year):
    s = daily[(daily["date"] >= pd.Timestamp(year - 1, 10, 1)) & (daily["date"] <= pd.Timestamp(year, 2, 28))]
    return s["PRECTOTCORR"].sum() / RAIN_RATIO


# Chiede una previsione a una delle 4 reti; "rows" è una tabella con gli input e la colonna cod_coltura.
def predict(network, rows):
    m = models[network]
    # Coltura -> colonne 0/1; reindex mette le colonne nell'ordine dell'addestramento e aggiunge a 0 quelle mancanti.
    X = pd.get_dummies(rows, columns=["cod_coltura"], dtype=float).reindex(columns=m["columns"], fill_value=0.0)
    # Stessa scala dell'addestramento, poi la rete prevede.
    return m["net"].predict(m["scaler"].transform(X))

## 4. Il calcolo completo per un campo
Per ogni coltura:
1. **Resa:** la rete prevede la resa relativa in ognuno dei 20 anni; la resa attesa è la mediana. In t/ha = resa relativa × resa mediana della coltura.
2. **Acqua:** la rete prevede l'irrigazione in ognuno dei 20 anni; anno normale = mediana, anno secco = l'80° percentile (serve almeno 1 anno su 5).
3. **Azoto:** la rete riceve la resa attesa in t/ha e le scelte dell'agricoltore.
4. **Stabilità:** la rete prevede il coefficiente di variazione della resa (%).

Poi la **classifica**: punteggio = resa relativa − peso dell'acqua × acqua anno normale / 100.
Le colture vietate dalla rotazione, con pH fuori dai limiti o il riso senza irrigazione vanno in fondo, escluse dalle 5.

In [5]:
# Distanza approssimata in km tra due punti (va bene per distanze piccole).
def distance_km(lat1, lon1, lat2, lon2):
    return 111 * np.sqrt((lat1 - lat2) ** 2 + ((lon1 - lon2) * np.cos(np.radians(lat1))) ** 2)


# Il calcolo completo. "choices" contiene le risposte dell'agricoltore; "log" è la funzione che scrive i messaggi.
def analyse_field(lat, lon, choices, log=print):
    # Il punto di pianura più vicino tra i 33 usati per addestrare, e la sua distanza.
    distances = distance_km(lat, lon, provinces["lat"], provinces["lon"])
    nearest = provinces.loc[distances.idxmin()]

    # Controllo 1: il punto deve essere vicino ai dati di addestramento (entro 60 km).
    if distances.min() > 60:
        raise ValueError("Il punto è fuori dalla Pianura Padana coperta dai dati: scegli un campo più vicino ai cerchietti verdi.")

    log("Leggo la quota del punto...")
    elevation = get_elevation(lat, lon)
    # Controllo 2: deve essere pianura. I punti di addestramento sono tutti sotto i 350 m.
    if elevation > 400:
        raise ValueError(f"Il punto è a {elevation:.0f} m: non è pianura. Le reti conoscono solo campi sotto i 400 m.")

    log("Cerco il suolo su SoilGrids...")
    soil, soil_source = get_soil(lat, lon, nearest)
    # Controllo 3: in mare la quota è esattamente 0 e SoilGrids non ha dati.
    if elevation == 0 and soil_source != "SoilGrids":
        raise ValueError("Il punto sembra in mare o in laguna: scegli un campo sulla terraferma.")

    log("Scarico 20 anni di clima da NASA POWER (circa 15 secondi)...")
    daily = download_climate(lat, lon, elevation)
    # Se l'agricoltore ha inserito la sua analisi del terreno, quei valori sostituiscono SoilGrids.
    if choices["lab"]:
        soil.update(choices["lab"])
        soil_source = "analisi del terreno dell'agricoltore + " + soil_source

    # Le reti conoscono solo i suoli dei 33 punti di addestramento: un valore fuori da quell'intervallo viene
    # riportato al limite più vicino (altrimenti le reti "inventano"), e lo segnaliamo all'agricoltore.
    soil_columns = ["ph", "argilla_pct", "sabbia_pct", "limo_pct", "carbonio_organico_g_kg", "azoto_g_kg"]
    low, high = provinces[soil_columns].min(), provinces[soil_columns].max()
    clipped = [c for c in soil_columns if not low[c] <= soil[c] <= high[c]]
    network_soil = {c: float(np.clip(soil[c], low[c], high[c])) for c in soil_columns}

    # Suolo e posizione: uguali per tutte le colture e tutti gli anni.
    place = dict(network_soil, lat=lat, lon=lon, quota_m=elevation)
    # Pioggia invernale mediana dei 20 anni, per l'azoto.
    rain_winter = np.median([winter_rain(daily, year) for year in YEARS])

    log("Le 4 reti neurali stanno calcolando...")
    results = []
    for code in CROPS:
        # Una riga per ogni anno: clima della stagione + suolo e posizione + coltura.
        years = pd.DataFrame([dict(season_climate(daily, code, year), **place, cod_coltura=code) for year in YEARS])

        # 1. Resa relativa nei 20 anni e resa attesa (mediana).
        relative = np.median(predict("yield", years))
        t_ha = relative * models["crop_median_t_ha"][code]

        # 2. Acqua nei 20 anni (in mm, mai negativa): anno normale e anno secco.
        water_rows = pd.DataFrame([dict(cycle_water(daily, code, year), cod_coltura=code) for year in YEARS])
        water = (predict("water", water_rows) * 100).clip(min=0)

        # 3. Azoto: una sola riga, con la resa attesa e le scelte dell'agricoltore.
        nitrogen_row = pd.DataFrame([dict(resa_t_ha=t_ha, pioggia_ott_feb_mm=rain_winter,
                                          dopo_medica=int(choices["previous"] == "LUCERNE"),
                                          sostanza_organica=choices["organic_matter"],
                                          residui_interrati=int(choices["residues"]), cod_coltura=code)])
        nitrogen = max(0.0, predict("nitrogen", nitrogen_row)[0] * 100)

        # 4. Stabilità: clima medio, variabilità del clima (_std), suolo e posizione.
        st = models["stability"]
        stability_row = pd.DataFrame([dict(years[st["climate"]].mean(),
                                           **years[st["climate_spread"]].std().add_suffix("_std"),
                                           **place, cod_coltura=code)])
        cv = predict("stability", stability_row)[0] * 10

        # Una riga di risultati per questa coltura.
        results.append(dict(code=code, crop=CROPS[code]["name"], relative=relative, t_ha=t_ha,
                            water_normal=np.median(water), water_dry=np.percentile(water, 80),
                            nitrogen=nitrogen, cv=cv))

    table = pd.DataFrame(results)
    # Punteggio: la resa relativa conta di più, l'acqua toglie punti (di più se non si può irrigare).
    weight = WATER_WEIGHT_IRRIGATED if choices["can_irrigate"] else WATER_WEIGHT_RAINFED
    table["score"] = table["relative"] - weight * table["water_normal"] / 100
    # Colture escluse: vietate dalla rotazione, pH fuori dai limiti, riso se non si può irrigare.
    table["excluded"] = [(code, choices["previous"]) in FORBIDDEN
                         or not CROPS[code]["ph"][2] <= soil["ph"] <= CROPS[code]["ph"][3]
                         or (code == "RICE" and not choices["can_irrigate"]) for code in table["code"]]
    # Ordina: prima le non escluse, poi per punteggio dal più alto.
    table = table.sort_values(["excluded", "score"], ascending=[True, False]).reset_index(drop=True)
    return table, soil, soil_source, elevation, clipped

## 5. Tabella-heatmap e consigli in parole semplici
Ogni colonna è colorata dal **verde** (meglio) al **rosso** (peggio): per la resa più alta è meglio, per acqua,
azoto e oscillazione più basso è meglio. I consigli nascono da **regole** scritte da noi a partire dai numeri,
non da un modello che potrebbe inventare.

In [6]:
# Mostra la tabella colorata delle prime 5 colture.
def show_table(table):
    top5 = table[~table["excluded"]].head(5)
    # Una tabella con nomi di colonna comprensibili e numeri arrotondati.
    shown = pd.DataFrame({
        "Coltura": top5["crop"],
        "Resa attesa (t/ha)": top5["t_ha"].round(1),
        "Resa rispetto alla media (%)": ((top5["relative"] - 1) * 100).round(0),
        "Acqua anno normale (mm)": top5["water_normal"].round(0),
        "Acqua anno secco (mm)": top5["water_dry"].round(0),
        "Azoto (kg N/ha)": top5["nitrogen"].round(0),
        "Oscillazione resa (CV %)": top5["cv"].round(0),
    }).set_index("Coltura")
    # Colori: RdYlGn va dal rosso al verde (alto = verde); RdYlGn_r al contrario (basso = verde).
    # La resa in t/ha non si colora: tra colture diverse non si confronta (la barbabietola fa 60 t/ha, la soia 3).
    styled = (shown.style
              .background_gradient(cmap="RdYlGn", subset=[shown.columns[1]])
              .background_gradient(cmap="RdYlGn_r", subset=list(shown.columns[2:]))
              .format("{:.0f}").format("{:.1f}", subset=[shown.columns[0]]))
    display(styled)


# Nomi leggibili delle colonne del suolo, per gli avvisi.
SOIL_NAMES = {"ph": "pH", "argilla_pct": "argilla", "sabbia_pct": "sabbia", "limo_pct": "limo",
              "carbonio_organico_g_kg": "carbonio organico", "azoto_g_kg": "azoto"}


# Scrive i consigli in parole semplici, uno per riga.
def advice(table, soil, choices, clipped):
    tips = []
    # Avviso se qualche valore del suolo era fuori da quelli visti in addestramento.
    if clipped:
        names = ", ".join(SOIL_NAMES[c] for c in clipped)
        tips.append(f"ℹ️ Alcuni valori del suolo ({names}) sono fuori da quelli che le reti hanno visto in addestramento: hanno usato il valore più vicino che conoscono, quindi i risultati sono meno affidabili.")
    previous = choices["previous"]
    for row in table[~table["excluded"]].head(5).itertuples():
        name = f"<b>{row.crop}</b>"
        # Resa: di quanto sopra o sotto la media della Pianura Padana.
        percent = (row.relative - 1) * 100
        if abs(percent) < 1:
            tips.append(f"{name}: qui rende come la media padana ({row.t_ha:.1f} t/ha).")
        else:
            tips.append(f"{name}: qui rende circa il {abs(percent):.0f}% {'in più' if percent > 0 else 'in meno'} della media padana ({row.t_ha:.1f} t/ha).")
        # Acqua: avviso se in un anno secco serve molta irrigazione.
        if row.water_dry > 250:
            tips.append(f"{name}: in un'estate secca come il 2022 servirebbero circa {row.water_dry:.0f} mm di irrigazione: valuta un'alternativa se l'acqua è scarsa.")
        # Azoto: le leguminose non ne hanno bisogno.
        if row.code == "SOYA":
            tips.append(f"{name}: è una leguminosa: non serve concime azotato e lascia azoto nel suolo per la coltura dopo.")
        # Stabilità: avviso se la resa cambia molto da un anno all'altro.
        if row.cv > 18:
            tips.append(f"{name}: la resa cambia molto da un anno all'altro (oscillazione {row.cv:.0f}%): è una scelta più rischiosa.")
        # Rotazione: avvisi per la coltura dell'anno scorso.
        if (row.code, previous) in WARNINGS:
            tips.append(f"{name}: {WARNINGS[(row.code, previous)]}.")
        # pH fuori dall'intervallo ottimale (ma dentro i limiti, altrimenti sarebbe esclusa).
        low, high = CROPS[row.code]["ph"][:2]
        if not low <= soil["ph"] <= high:
            tips.append(f"{name}: pH del suolo {soil['ph']:.1f}, fuori dall'ottimo della coltura ({low}-{high}).")
    # Le colture escluse e il motivo.
    for row in table[table["excluded"]].itertuples():
        if (row.code, previous) in FORBIDDEN:
            reason = FORBIDDEN[(row.code, previous)]
        elif row.code == "RICE" and not choices["can_irrigate"]:
            reason = "senza irrigazione non si può fare la risaia"
        else:
            reason = f"pH del suolo {soil['ph']:.1f} fuori dai limiti della coltura"
        tips.append(f"❌ <b>{row.crop}</b> esclusa: {reason}.")
    # Un elenco puntato in HTML.
    display(HTML("<ul>" + "".join(f"<li>{t}</li>" for t in tips) + "</ul>"))


# Calcola e mostra tutto per un campo: intestazione, tabella, consigli e avviso.
def show_results(lat, lon, choices):
    try:
        table, soil, soil_source, elevation, clipped = analyse_field(lat, lon, choices)
    except ValueError as problem:
        # Il punto non va bene (fuori pianura, in mare o lontano dai dati): lo spieghiamo e ci fermiamo.
        print("⚠️", problem)
        return
    # Intestazione: coordinate, quota e suolo usato.
    display(HTML(f"<h3>🌱 FarmerBud: le 5 colture più adatte al tuo campo</h3>"
                 f"<p>{lat:.4f}, {lon:.4f} · {elevation:.0f} m · suolo da {soil_source} · "
                 f"pH {soil['ph']:.1f} · argilla {soil['argilla_pct']:.0f}% · sabbia {soil['sabbia_pct']:.0f}%</p>"))
    show_table(table)
    display(HTML("<h3>Consigli</h3>"))
    advice(table, soil, choices, clipped)
    display(HTML("<p><i>⚠️ Prototipo per un hackathon, non è un consiglio agronomico. Prima di seminare parlane con un tecnico.</i></p>"))

## 6. L'interfaccia: mappa e domande all'agricoltore
- I **cerchietti verdi** sono i 33 punti di pianura usati per addestrare le reti; il **rettangolo** è l'area coperta.
- **Clicca sul tuo campo**: compare il segnaposto e le coordinate vengono scritte sotto la mappa.
- Le domande sono facoltative. Per l'analisi del terreno lascia vuoto ciò che non sai.

In [7]:
# La mappa, centrata sulla Pianura Padana; non si può allontanare troppo lo zoom.
field_map = Map(center=(45.1, 10.3), zoom=7, min_zoom=7, scroll_wheel_zoom=True, layout=widgets.Layout(height="450px"))

# Il rettangolo che mostra l'area coperta dai dati.
field_map.add(Rectangle(bounds=((44.0, 7.3), (46.0, 12.8)), color="green", fill_opacity=0.03, weight=1))

# Un cerchietto verde per ognuno dei 33 punti di pianura.
for p in provinces.itertuples():
    field_map.add(CircleMarker(location=(p.lat, p.lon), radius=4, color="green", fill_opacity=0.7, weight=1))

# Il segnaposto del campo: all'inizio non c'è, compare al primo clic.
pin = Marker(location=(45.1, 10.3), draggable=False)

# Il punto scelto: all'inizio nessuno.
chosen = {"lat": None, "lon": None}

# Testo sotto la mappa con le coordinate.
coordinates = widgets.HTML("👆 Clicca sulla mappa per scegliere il tuo campo.")


# Questa funzione viene chiamata dalla mappa a ogni interazione (clic, movimento del mouse...).
def on_map_click(**event):
    # Ci interessano solo i clic.
    if event.get("type") != "click":
        return
    # Le coordinate del clic: [latitudine, longitudine].
    lat, lon = event["coordinates"]
    chosen["lat"], chosen["lon"] = lat, lon
    # Sposta il segnaposto sul clic e lo aggiunge alla mappa se non c'era.
    pin.location = (lat, lon)
    if pin not in field_map.layers:
        field_map.add(pin)
    coordinates.value = f"📍 Campo scelto: <b>{lat:.4f}, {lon:.4f}</b>"


# Collega la funzione alla mappa.
field_map.on_interaction(on_map_click)

# Le domande all'agricoltore (tutte facoltative).
style = {"description_width": "260px"}
previous_crop = widgets.Dropdown(options=PREVIOUS_OPTIONS, description="Coltura dell'anno scorso", style=style)
can_irrigate = widgets.Checkbox(value=True, description="Posso irrigare (pozzo, consorzio)", style=style)
organic_matter = widgets.Dropdown(options=[("normale / non so", 0), ("bassa", -1), ("alta", 1)],
                                  description="Sostanza organica del suolo", style=style)
residues = widgets.Checkbox(value=False, description="Interro paglie o stocchi della coltura prima", style=style)

# Analisi del terreno: caselle di testo, vuote = "non lo so".
lab_ph = widgets.Text(description="pH", placeholder="es. 7,2", style=style)
lab_clay = widgets.Text(description="Argilla (%)", placeholder="es. 25", style=style)
lab_sand = widgets.Text(description="Sabbia (%)", placeholder="es. 30", style=style)
lab_silt = widgets.Text(description="Limo (%)", placeholder="es. 45", style=style)
lab_om = widgets.Text(description="Sostanza organica (%)", placeholder="es. 2,0", style=style)
lab_n = widgets.Text(description="Azoto totale (g/kg)", placeholder="es. 1,4", style=style)

# Il pulsante e lo spazio dove compaiono i risultati.
button = widgets.Button(description="Calcola", button_style="success", icon="seedling")
results_area = widgets.Output()


# Legge un numero da una casella di testo; accetta la virgola; se è vuota restituisce None.
def read_number(box):
    text = box.value.strip().replace(",", ".")
    return float(text) if text else None


# Raccoglie le risposte dell'agricoltore in un dizionario.
def read_choices():
    lab = {}
    for key, box in [("ph", lab_ph), ("argilla_pct", lab_clay), ("sabbia_pct", lab_sand), ("limo_pct", lab_silt), ("azoto_g_kg", lab_n)]:
        if read_number(box) is not None:
            lab[key] = read_number(box)
    # Il laboratorio dà la sostanza organica in %; le reti vogliono il carbonio organico in g/kg: SO ÷ 1,724 × 10.
    if read_number(lab_om) is not None:
        lab["carbonio_organico_g_kg"] = read_number(lab_om) / 1.724 * 10
    return {"previous": previous_crop.value, "can_irrigate": can_irrigate.value,
            "organic_matter": organic_matter.value, "residues": residues.value, "lab": lab}


# Cosa succede quando si preme "Calcola".
def on_button_click(_):
    # Tutto quello che viene scritto finisce nello spazio dei risultati, sotto il pulsante.
    with results_area:
        # Cancella i risultati precedenti.
        results_area.clear_output()
        if chosen["lat"] is None:
            print("Prima clicca sulla mappa per scegliere il campo.")
            return
        show_results(chosen["lat"], chosen["lon"], read_choices())


# Collega la funzione al pulsante.
button.on_click(on_button_click)

# Mette tutto in colonna e lo mostra: mappa, coordinate, domande, analisi del terreno, pulsante, risultati.
display(widgets.VBox([
    widgets.HTML("<h2>🌱 FarmerBud</h2>"),
    field_map, coordinates,
    widgets.HTML("<b>Il tuo campo (facoltativo)</b>"),
    previous_crop, can_irrigate, organic_matter, residues,
    widgets.HTML("<b>Analisi del terreno (facoltativa, lascia vuoto ciò che non sai)</b>"),
    lab_ph, lab_clay, lab_sand, lab_silt, lab_om, lab_n,
    button, results_area,
]))

## 7. Esempio già calcolato (senza mappa)
Su GitHub la mappa interattiva non si vede, quindi qui FarmerBud è eseguito su un **campo di esempio** nella pianura
cremonese, vicino a Pizzighettone (45,15 N, 9,95 E). L'agricoltore l'anno scorso ha coltivato **mais**, può irrigare
e interra gli stocchi. Puoi cambiare coordinate e scelte e rieseguire la cella.

In [8]:
# Le scelte dell'agricoltore di esempio: mais l'anno scorso, irrigazione sì, sostanza organica normale, stocchi interrati, nessuna analisi.
example_choices = {"previous": "MAIZE", "can_irrigate": True, "organic_matter": 0, "residues": True, "lab": {}}

# Calcola e mostra i risultati per il campo di esempio.
show_results(45.15, 9.95, example_choices)

Leggo la quota del punto...


Cerco il suolo su SoilGrids...


Scarico 20 anni di clima da NASA POWER (circa 15 secondi)...


Le 4 reti neurali stanno calcolando...


,Resa attesa (t/ha),Resa rispetto alla media (%),Acqua anno normale (mm),Acqua anno secco (mm),Azoto (kg N/ha),Oscillazione resa (CV %)
Coltura,,,,,,
frumento tenero,5.9,4,20,35,195,13
orzo,5.3,-0,3,12,157,15
soia,3.4,6,218,286,0,16
colza,2.8,-6,6,13,176,17
frumento duro,5.2,-6,18,33,194,16
